In [11]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from pathlib import Path

## Pdf loader
Langchain pyMuPDFLoader is used to load the pdf file from the directory


In [2]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)

pdf_doc = dir_loader.load();

pdf_doc

[Document(metadata={'producer': 'Acrobat Distiller 11.0 (Windows); modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'LaTeX with hyperref package', 'creationdate': '2023-07-21T10:43:30+05:30', 'source': '../data/pdf/qayyum.pdf', 'file_path': '../data/pdf/qayyum.pdf', 'total_pages': 11, 'format': 'PDF 1.4', 'title': 'High-Density Electroencephalography and Speech Signal Based Deep Framework for Clinical Depression Diagnosis', 'author': '', 'subject': 'IEEE/ACM Transactions on Computational Biology and Bioinformatics;2023;20;4;10.1109/TCBB.2023.3257175', 'keywords': '', 'moddate': '2023-08-08T21:28:20-04:00', 'trapped': '', 'modDate': "D:20230808212820-04'00'", 'creationDate': "D:20230721104330+05'30'", 'page': 0}, page_content='IEEE/ACM TRANSACTIONS ON COMPUTATIONAL BIOLOGY AND BIOINFORMATICS, VOL. 20, NO. 4, JULY/AUGUST 2023\n2587\nHigh-Density Electroencephalography and Speech\nSignal Based Deep Framework for Clinical\nDepression Diagnosis\nAbdul Q

### RAG retrieval Pipeline

In [12]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf")

Found 1 PDF files to process

Processing: qayyum.pdf


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/FirstChar': 30, '/Subtype': '/Type1', '/BaseFont': '/MGOMJI+Times-Roman', '/FontDescriptor': IndirectObject(43, 0, 133971152135440), '/Type': '/Font', '/Encoding': IndirectObject(44, 0, 133971152135440), '/LastChar': 233, '/Widths': [556, 556, 0, 0, 0, 0, 0, 833, 778, 0, 333, 333, 0, 564, 250, 333, 250, 278, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 278, 278, 0, 0, 0, 0, 921, 722, 667, 667, 722, 611, 556, 722, 722, 333, 389, 722, 611, 889, 722, 722, 556, 722, 667, 556, 611, 722, 722, 944, 722, 722, 611, 333, 0, 333, 0, 500, 0, 444, 500, 444, 500, 444, 333, 500, 500, 278, 278, 500, 278, 778, 500, 500, 500, 500, 333, 389, 278, 500, 500, 722, 500, 500, 444, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 333, 333, 444, 444, 0, 500, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 760, 0, 0, 0, 333, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

  ✓ Loaded 11 pages

Total documents loaded: 11


In [13]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [14]:
chunks=split_documents(all_pdf_documents)
chunks

Split 11 documents into 66 chunks

Example chunk:
Content: IEEE/ACM TRANSACTIONS ON COMPUTATIONAL BIOLOGY AND BIOINFORMATICS, VOL. 20, NO. 4, JUL Y/AUGUST 2023 2587
High-Density Electroencephalography and Speech
Signal Based Deep Framework for Clinical
Depres...
Metadata: {'producer': 'Acrobat Distiller 11.0 (Windows); modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'LaTeX with hyperref package', 'creationdate': '2023-07-21T10:43:30+05:30', 'moddate': '2023-08-08T21:28:20-04:00', 'ieee article id': '10068766', 'ieee issue id': '10214206', 'subject': 'IEEE/ACM Transactions on Computational Biology and Bioinformatics;2023;20;4;10.1109/TCBB.2023.3257175', 'ieee publication id': '8857', 'title': 'High-Density Electroencephalography and Speech Signal Based Deep Framework for Clinical Depression Diagnosis', 'source': '../data/pdf/qayyum.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'qayyum.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Acrobat Distiller 11.0 (Windows); modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'LaTeX with hyperref package', 'creationdate': '2023-07-21T10:43:30+05:30', 'moddate': '2023-08-08T21:28:20-04:00', 'ieee article id': '10068766', 'ieee issue id': '10214206', 'subject': 'IEEE/ACM Transactions on Computational Biology and Bioinformatics;2023;20;4;10.1109/TCBB.2023.3257175', 'ieee publication id': '8857', 'title': 'High-Density Electroencephalography and Speech Signal Based Deep Framework for Clinical Depression Diagnosis', 'source': '../data/pdf/qayyum.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'qayyum.pdf', 'file_type': 'pdf'}, page_content='IEEE/ACM TRANSACTIONS ON COMPUTATIONAL BIOLOGY AND BIOINFORMATICS, VOL. 20, NO. 4, JUL Y/AUGUST 2023 2587\nHigh-Density Electroencephalography and Speech\nSignal Based Deep Framework for Clinical\nDepression Diagnosis\nAbdul Qayyum , Imran Razzak , M. T

### Embeddings and Vector store

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/home/mitnick/Code/RAG/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1573.70it/s]


Model loaded successfully. Embedding dimension: 384


/tmp/ipykernel_77802/2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [16]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [19]:
### convert the text to the embeddings
texts = [doc.page_content for doc in chunks]

### Generate the embeddings
embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)


Generating embeddings for 66 texts...


Batches: 100%|██████████| 3/3 [00:03<00:00,  1.32s/it]


Generated embeddings with shape: (66, 384)
Adding 66 documents to vector store...
Successfully added 66 documents to vector store
Total documents in collection: 66


In [20]:
### Retrieval pipeline from the VectorStore

In [21]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 10, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [22]:
rag_retriever.retrieve("What is Depression")

Retrieving documents for query: 'What is Depression'
Top K: 10, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.03it/s]

Generated embeddings with shape: (1, 384)
Retrieved 9 documents (after filtering)


[{'id': 'doc_0f2ceddc_4',
  'content': 'as medication, drugs, stress, genetic vulnerability, and medical\nconditions. While we all feel moody, sad, or low, some feel\nintense and for a more extended period, and it may be without\nany apparent reason. Depression ranges in seriousness from\nmild symptoms to temporary episodes of sadness and severe\npersistent depression. Clinical depression is the more-severe\nform of depression, also known as major depressive disorder\nor major depression. It is marked by a depressed mood most of\nthe day, sometimes particularly in the morning, and a loss of\ninterest in performing routine activities and relationships.\nDepression is a threatening life condition and impacts our\nmental and physical health. Researchers argue that continuous\nchallenges in employment, illness, abusive relationships, lone-\nliness, or longer working hours may cause depression. Even\nthough the causes of depression have been actively explored,\nthere is still much to be kno

### RAG pipeline - VectorDB to LLM Answer Generation

In [23]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

None


In [29]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage